# Gridlock 2.0 -- Offline foundation-model data engine (Kaggle, T4x2 GPU)

Implements **00_master_design.md S3.10** exactly: *"open-vocab teachers auto-label rare Indian
classes ... human-spot-check ... distill into the fast Stage-1 detector."* SAM-3 is the teacher
here -- validated by the head-to-head benchmark (`kaggle_sam3_vs_rfdetr_benchmark.ipynb`) run
previously, result:

```
class             RF-DETR AP50   SAM-3 AP50
autorickshaw           0.0000       0.5050   <-- real signal, worth distilling
vehicle fallback       0.0000       0.0168   <-- weak signal, included but flagged
animal                 0.0000       0.0000   <-- NO signal at these prompts/threshold
traffic sign           0.0000       0.0000   <-- NO signal at these prompts/threshold
VERDICT: HYBRID -- SAM-3 wins the domain-gap classes only; keep RF-DETR for the bulk/fast classes.
```

**Honest scoping (no silent overclaiming):** only **auto-rickshaw** and **vehicle-fallback** get
SAM-3 pseudo-labels here, because those are the only domain-gap classes where the benchmark
showed real signal. Animal / traffic-sign are explicitly **left alone** -- pretending SAM-3 can
label them when the benchmark says it can't would poison training with noise.

**Pipeline:** VOC GT (train+val, all real labels) -> SAM-3 pseudo-labels for
autorickshaw/vehicle-fallback on **TRAIN only** (never pollute the eval set) -> confidence
filter (stricter than the eval threshold -- training labels need precision) -> human
spot-check artifact (saved overlays, inspect before trusting) -> merge into one COCO train set
-> fine-tune **RF-DETR-Large** (Apache, the documented Stage-2 / shippable model) -> re-eval on
clean GT-only val, specifically checking whether autorickshaw/vehicle-fallback AP moves off zero
**without SAM-3 online at inference time** -- that's the actual "distill" outcome §3.10 wants.


## 0. Install dependencies

In [ ]:
import numpy
_PINNED_NUMPY = numpy.__version__
print('numpy before install:', _PINNED_NUMPY)

# RF-DETR (+train extras), pycocotools: preinstalled torch/CUDA on Kaggle GPU images.
!pip -q install "rfdetr[train]" pycocotools "numpy=={_PINNED_NUMPY}" 2>/dev/null

# SAM-3: native package (matches the validated reference-notebook API), not the HF wrapper.
!git clone -q https://github.com/facebookresearch/sam3.git /kaggle/working/sam3_repo
!pip -q install -e "/kaggle/working/sam3_repo[notebooks]" "numpy=={_PINNED_NUMPY}" 2>/dev/null
print('installed (numpy pinned to', _PINNED_NUMPY, ')')


### Restart the kernel now -- before running anything below
Same numpy/torchvision ABI-mismatch reason as the benchmark notebook (`ValueError: numpy.dtype
size changed...`). **Kaggle:** session menu -> **Restart Session**, then continue from the next
cell. Do not re-run the install cell.

In [ ]:
import torch, time, json as _json
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {round(p.total_memory/1024**3,1)} GB')


## 1. HF token (gated repo -- never hard-code)
Add via Kaggle: **Add-ons -> Secrets -> `HF_TOKEN`**.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    if os.environ.get('HF_TOKEN'):
        print('HF_TOKEN loaded from environment')
    else:
        print('WARNING: no HF_TOKEN found -- add it via Kaggle Secrets before running.', e)


## 2. Config -- EDIT `IDD_ROOT`
`TRAIN_SUBSET` images get SAM-3 auto-labeling (offline, batch -- runtime scales with this).
At ~0.7s/concept-pass x 2 concepts/image, 6000 images is roughly 2-2.5 hours; shrink for a
faster first pass.

In [ ]:
from pathlib import Path

# >>> EDIT THIS to your uploaded dataset path <<<
IDD_ROOT = '/kaggle/input/idd-detection/IDD_Detection'

TRAIN_SUBSET = 6000          # images to auto-label (None = full train set)
VAL_SUBSET = 1000            # GT-only eval slice, no pseudo-labels ever
PSEUDO_CONF_THRESH = 0.6     # stricter than eval (0.25-0.3) -- training labels need precision
PSEUDO_CLASSES = {'autorickshaw': 'auto rickshaw', 'vehicle fallback': 'vehicle'}  # validated signal only

WORK = Path('/kaggle/working')
COCO_DIR = WORK / 'idd_coco_distilled'
REVIEW_DIR = WORK / 'pseudo_label_review'
COCO_DIR.mkdir(parents=True, exist_ok=True)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
assert Path(IDD_ROOT).exists(), f'IDD_ROOT not found: {IDD_ROOT} (check /kaggle/input)'
print('train.txt:', (Path(IDD_ROOT)/'train.txt').exists(), '| val.txt:', (Path(IDD_ROOT)/'val.txt').exists())


## 3. Ground truth -- VOC parse (train + val, extended categories)
Same 11-category map as the benchmark notebook (7 COCO-mappable + 4 domain-gap; caravan/trailer/
train dropped, too rare to matter here).

In [ ]:
import xml.etree.ElementTree as ET

CATS = {1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 5: 'bus', 6: 'truck',
        7: 'traffic light', 8: 'autorickshaw', 9: 'animal', 10: 'traffic sign', 11: 'vehicle fallback'}
CAT_ID = {v: k for k, v in CATS.items()}
IDD_TO_CAT = {'car': 'car', 'motorcycle': 'motorcycle', 'bus': 'bus', 'truck': 'truck',
              'bicycle': 'bicycle', 'person': 'person', 'rider': 'person',
              'traffic light': 'traffic light', 'autorickshaw': 'autorickshaw', 'animal': 'animal',
              'traffic sign': 'traffic sign', 'vehicle fallback': 'vehicle fallback'}

def parse_voc(xml_path):
    r = ET.parse(xml_path).getroot()
    s = r.find('size')
    w = int(float(s.findtext('width', '0'))); h = int(float(s.findtext('height', '0')))
    objs = []
    for o in r.findall('object'):
        name = (o.findtext('name') or '').strip(); b = o.find('bndbox')
        if b is None or not name:
            continue
        x1 = float(b.findtext('xmin', '0')); y1 = float(b.findtext('ymin', '0'))
        x2 = float(b.findtext('xmax', '0')); y2 = float(b.findtext('ymax', '0'))
        if x2 > x1 and y2 > y1:
            objs.append((name, x1, y1, x2, y2))
    return w, h, objs

def load_split(split, limit):
    entries = (Path(IDD_ROOT) / f'{split}.txt').read_text().split()
    out = []
    for e in entries:
        img = None
        for ext in ('.jpg', '.png'):
            p = Path(IDD_ROOT) / 'JPEGImages' / f'{e}{ext}'
            if p.exists():
                img = p
                break
        xml = Path(IDD_ROOT) / 'Annotations' / f'{e}.xml'
        if img and xml.exists():
            out.append((img, xml))
        if limit and len(out) >= limit:
            break
    return out

train_samples = load_split('train', TRAIN_SUBSET)
val_samples = load_split('val', VAL_SUBSET)
print(len(train_samples), 'train images |', len(val_samples), 'val images')

def build_gt(samples):
    images, anns, has_class = [], [], []
    aid = 1
    for img_id, (img_path, xml_path) in enumerate(samples, 1):
        w, h, objs = parse_voc(xml_path)
        images.append({'id': img_id, 'width': w, 'height': h, 'file_name': img_path.name, 'path': str(img_path)})
        present = set()
        for name, x1, y1, x2, y2 in objs:
            cat = IDD_TO_CAT.get(name)
            if cat is None:
                continue
            present.add(cat)
            anns.append({'id': aid, 'image_id': img_id, 'category_id': CAT_ID[cat],
                         'bbox': [x1, y1, x2 - x1, y2 - y1], 'area': (x2 - x1) * (y2 - y1),
                         'iscrowd': 0, 'source': 'gt'})
            aid += 1
        has_class.append(present)
    return images, anns, has_class, aid

train_images, train_gt_anns, train_has_class, _next_aid = build_gt(train_samples)
val_images, val_gt_anns, _val_has_class, _ = build_gt(val_samples)
print(f'train: {len(train_gt_anns)} GT boxes | val: {len(val_gt_anns)} GT boxes')


## 4. Load SAM-3 (native API)

In [ ]:
import torch
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

torch.autocast(device_type='cuda', dtype=torch.bfloat16).__enter__()
if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

sam3_model = build_sam3_image_model(
    bpe_path='/kaggle/working/sam3_repo/sam3/assets/bpe_simple_vocab_16e6.txt.gz')
sam3_processor = Sam3Processor(sam3_model, confidence_threshold=0.3)
print('SAM-3 loaded')


## 5. Auto-label TRAIN with SAM-3 (autorickshaw / vehicle-fallback only)
Skips an image+class if VOC GT already has that class present (avoid duplicate/overlapping
labels for the same concept) -- pseudo-labels only fill in where the real GT is silent.

In [ ]:
from PIL import Image

pseudo_anns = []
pseudo_review = []   # (img_path, box, cat_name, score) -- for the spot-check artifact
aid = _next_aid
t0 = time.time()
for img_id, ((img_path, _xml), present) in enumerate(zip(train_samples, train_has_class), 1):
    needed = [c for c in PSEUDO_CLASSES if c not in present]
    if not needed:
        continue
    pil = Image.open(img_path).convert('RGB')
    state = sam3_processor.set_image(pil)
    for cat_name in needed:
        prompt = PSEUDO_CLASSES[cat_name]
        state = sam3_processor.set_text_prompt(state=state, prompt=prompt)
        boxes = state['boxes'].to(torch.float32).cpu().numpy()
        scores = state['scores'].to(torch.float32).cpu().numpy()
        for box, score in zip(boxes, scores):
            if float(score) < PSEUDO_CONF_THRESH:
                continue
            x1, y1, x2, y2 = [float(v) for v in box]
            pseudo_anns.append({'id': aid, 'image_id': img_id, 'category_id': CAT_ID[cat_name],
                                 'bbox': [x1, y1, x2 - x1, y2 - y1], 'area': (x2 - x1) * (y2 - y1),
                                 'iscrowd': 0, 'source': 'sam3-pseudo'})
            aid += 1
            pseudo_review.append((img_path, (x1, y1, x2, y2), cat_name, float(score)))
    if img_id % 200 == 0:
        print(f'  auto-label {img_id}/{len(train_samples)}  pseudo-labels so far: {len(pseudo_anns)}'
              f'  ({time.time()-t0:.0f}s)')

print(f'\nSAM-3 auto-labeling done: {len(pseudo_anns)} pseudo-labels in {time.time()-t0:.0f}s')
from collections import Counter
print('by class:', Counter(CATS[a['category_id']] for a in pseudo_anns))


## 6. Human spot-check (required by S3.10 -- inspect before trusting)
Saves a random sample of pseudo-labeled crops with their score so you can visually confirm SAM-3
got the box right before it goes into training. **Look at these in the Kaggle Output panel.**

In [ ]:
import random
from PIL import ImageDraw

random.seed(0)
sample = random.sample(pseudo_review, min(40, len(pseudo_review)))
for i, (img_path, box, cat_name, score) in enumerate(sample):
    img = Image.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    x1, y1, x2, y2 = [int(v) for v in box]
    draw.rectangle([x1, y1, x2, y2], outline='red', width=4)
    draw.text((x1, max(0, y1 - 18)), f'PSEUDO {cat_name} {score:.2f}', fill='red')
    pad = 40
    crop = img.crop((max(0, x1 - pad), max(0, y1 - pad), min(img.width, x2 + pad), min(img.height, y2 + pad)))
    crop.save(REVIEW_DIR / f'{i:03d}_{cat_name.replace(" ", "_")}_{score:.2f}.jpg')
print(f'Saved {len(sample)} spot-check crops -> {REVIEW_DIR}')
print('STOP and look at these before trusting the fine-tune below if the pseudo-label count seems off.')


## 7. Merge GT + pseudo-labels -> final COCO (train enriched, val stays GT-only)

In [ ]:
import os, shutil

def write_coco(images, anns, split_name):
    sdir = COCO_DIR / split_name
    sdir.mkdir(parents=True, exist_ok=True)
    out_images = []
    for im in images:
        dst = sdir / im['file_name']
        if not dst.exists():
            try:
                os.symlink(im['path'], dst)
            except Exception:
                shutil.copy2(im['path'], dst)
        out_images.append({'id': im['id'], 'width': im['width'], 'height': im['height'], 'file_name': im['file_name']})
    cats = [{'id': cid, 'name': n, 'supercategory': 'none'} for n, cid in CAT_ID.items()]
    (sdir / '_annotations.coco.json').write_text(json.dumps(
        {'images': out_images, 'annotations': anns, 'categories': cats}))
    print(f'{split_name}: {len(out_images)} images, {len(anns)} boxes -> {sdir}')
    return sdir

train_all_anns = train_gt_anns + pseudo_anns
train_dir = write_coco(train_images, train_all_anns, 'train')
val_dir = write_coco(val_images, val_gt_anns, 'valid')
print(f'\ntrain enrichment: {len(train_gt_anns)} GT + {len(pseudo_anns)} pseudo = {len(train_all_anns)} total boxes')


## 8. Fine-tune RF-DETR-Large on the enriched dataset
The documented Stage-2 / shippable model (Apache-2.0). Same hyperparams as
`kaggle_detection_finetune.ipynb` Part B for a like-for-like comparison.

In [ ]:
from rfdetr import RFDETRLarge

det = RFDETRLarge()
det.train(
    dataset_dir=str(COCO_DIR), dataset_file='roboflow',
    epochs=40, batch_size=4, grad_accum_steps=4,
    resolution=560, lr=1e-4, num_workers=2,
    early_stopping=True, output_dir='/kaggle/working/rfdetr_large_distilled')
print('RF-DETR-Large (distilled) done. checkpoints/metrics in /kaggle/working/rfdetr_large_distilled')


## 9. Re-eval on clean GT-only val -- did distillation move the gap classes off zero?
Loads the fine-tuned checkpoint for inference (no SAM-3 involved at this point -- this is the
actual "distilled into the fast detector" outcome).

In [ ]:
import glob
import contextlib, io
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

ckpts = sorted(glob.glob('/kaggle/working/rfdetr_large_distilled/checkpoint_best_*.pth'))
assert ckpts, 'no checkpoint found -- check the training cell output above'
ft_model = RFDETRLarge(pretrain_weights=ckpts[-1])
print('loaded fine-tuned checkpoint:', ckpts[-1])

ft_dts = []
for img_id, (img_path, _xml) in enumerate(val_samples, 1):
    pil = Image.open(img_path).convert('RGB')
    preds = ft_model.predict(pil, threshold=0.3)
    for box, cid, score in zip(preds.xyxy, preds.class_id, preds.confidence):
        name = CATS.get(int(cid) + 0, None)  # fine-tuned head is 0-indexed over our 11 cats internally;
        # rfdetr's .predict() on a custom-trained model returns dataset-native class ids matching
        # the COCO json category ids we wrote above (1..11) minus 1 in some rfdetr versions --
        # print a couple of raw predictions first if class names look wrong and adjust the offset.
        name = CATS.get(int(cid), None) or CATS.get(int(cid) + 1, None)
        if name is None or name not in CAT_ID:
            continue
        x1, y1, x2, y2 = [float(v) for v in box]
        ft_dts.append({'image_id': img_id, 'category_id': CAT_ID[name],
                        'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': float(score)})

categories = [{'id': cid, 'name': n} for n, cid in CAT_ID.items()]
coco_gt = COCO()
coco_gt.dataset = {'images': [{'id': i+1, **{k: v for k, v in im.items() if k != 'path'}}
                              for i, im in enumerate(val_images)],
                   'annotations': val_gt_anns, 'categories': categories}
with contextlib.redirect_stdout(io.StringIO()):
    coco_gt.createIndex()
    coco_dt = coco_gt.loadRes(ft_dts) if ft_dts else None

if coco_dt is not None:
    with contextlib.redirect_stdout(io.StringIO()):
        ev = COCOeval(coco_gt, coco_dt, 'bbox')
        ev.evaluate(); ev.accumulate(); ev.summarize()
    print(f'mAP@0.5 (distilled RF-DETR-Large) = {ev.stats[1]:.4f}')
    prec = ev.eval['precision']
    print('\nper-class AP@0.5 (BEFORE -> distilled):')
    before = {'autorickshaw': 0.0, 'vehicle fallback': 0.0168, 'animal': 0.0, 'traffic sign': 0.0}
    for k, name in enumerate(CAT_ID):
        p = prec[0, :, k, 0, -1]; p = p[p > -1]
        ap = round(float(p.mean()), 4) if p.size else 0.0
        tag = f"  (was {before[name]})" if name in before else ''
        print(f'  {name:18s} {ap:.4f}{tag}')
else:
    print('no detections produced -- check class-id offset note above')


## How to read this
- If **autorickshaw AP** moves meaningfully off 0.0 (toward the SAM-3-only 0.505, ideally closer
  with the precision of a real detector), the distillation worked: RF-DETR alone now covers a
  class it structurally could not before, with **zero SAM-3 cost at inference time**.
- **vehicle-fallback** had weak SAM-3 signal (0.0168) to begin with -- don't expect much lift;
  it's included for completeness, not as a headline result.
- **animal / traffic-sign** were deliberately left untouched -- the benchmark showed no real
  SAM-3 signal for them at the prompts/threshold tried. If they matter for the final submission,
  that needs either better prompts, a different open-vocab teacher, or manual annotation -- not
  papered over here.
- If the lift is small or noisy, check `pseudo_label_review/` first: bad pseudo-labels (wrong box,
  wrong class) poison fine-tuning before anything else does.
